In [2]:
!git log --oneline

6290fc4 (HEAD -> main) Add notebook: enviroment setup and version check
eeb2c6c Install commit


# MOP 3231 · Week 2 Lab / Task 1
## Data Preprocessing & Cleaning

Student: Nuray Aben
ID: 2653
Date: 27.09.2026

In [4]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import sklearn

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.7
NumPy: 2.0.2
pandas: 2.3.3
matplotlib: 3.9.4
scikit-learn: 1.6.1


In [9]:
raw = pd.read_csv("data/almaty_apartments_raw.csv")
YOUR_ID = 2653
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=YOUR_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)
print("raw.shape:", raw.shape)
print("df.shape:", df.shape)
print("уникальных listing_id в raw:", raw["listing_id"].nunique())
print("уникальных listing_id в df:", df["listing_id"].nunique())
print("exact duplicates:", df.duplicated().sum())
print("reposts:", df.drop_duplicates()["listing_id"].duplicated().sum())

raw.shape: (1284, 11)
df.shape: (1160, 11)
уникальных listing_id в raw: 1200
уникальных listing_id в df: 1080
exact duplicates: 35
reposts: 45


raw это у нас весь исходной файл там у нас 1284 строк и 11 колонок
df рабочая таюлица после которого я взяла случайные 90%, 1169 строк и 11 колонок
в исходном файле уникальных listing_id 1200 разных объвлений получается остальные 84 повторные обьявления
в рабочем датасете уникальных id 1080 остальные 80 повторы
пропорция та же 1080/1200=0.9=90%


In [10]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

df["listed_date"] = pd.to_datetime(df["listed_date"])
df = df.sort_values("listed_date").drop_duplicates("listing_id", keep="last").reset_index(drop=True)

after = len(df)
print("before:", before, "after:", after, "removed:", before - after)
print("уникальных listing_id:", df["listing_id"].nunique())

before: 1160 after: 1080 removed: 80
уникальных listing_id: 1080


удаляем дубликаты так как уникальные значения они не должны повторятся

In [11]:
print(df["district"].value_counts())

district
Medeu            132
Bostandyk        125
Turksib          120
Auezov           116
Zhetysu          114
Almaly           109
Alatau           107
Nauryzbay        100
medeu              9
turksib            9
auezov             7
 Auezov            7
 Zhetysu           6
bostandyk          6
Nauryzbayskiy      6
Alatauskiy         6
Наурызбайский      6
Auezovskiy         6
AUEZOV             6
BOSTANDYK          5
nauryzbay          5
Bostandykskiy      5
Медеуский          4
ZHETYSU            4
 Medeu             4
 Alatau            4
TURKSIB            4
almaly             4
ALMALY             4
Zhetysuskiy        3
Бостандыкский      3
Турксибский        3
 Turksib           3
 Almaly            3
Almalinskiy        3
NAURYZBAY          3
 Bostandyk         3
MEDEU              3
ALATAU             3
Turksibskiy        2
Ауэзовский         2
Medeuskiy          1
Алмалинский        1
zhetysu            1
 Nauryzbay         1
alatau             1
Жетысуский         1
Name

в датасете один и тот же район записан по разному: 8 правильных названий :(Medeu, Bostandyk, Turksib, Auezov, Zhetysu, Almaly, Alatau, Nauryzbay — у них большие числа) и ещё много кривых вариантов тех же районов с маленькой буквы 

In [12]:
df["district"] = df["district"].str.strip().str.casefold()

mapping = {
    "alatau": "Alatau", "alatauskiy": "Alatau", "алатауский": "Alatau",
    "almaly": "Almaly", "almalinskiy": "Almaly", "алмалинский": "Almaly",
    "auezov": "Auezov", "auezovskiy": "Auezov", "ауэзовский": "Auezov",
    "bostandyk": "Bostandyk", "bostandykskiy": "Bostandyk", "бостандыкский": "Bostandyk",
    "medeu": "Medeu", "medeuskiy": "Medeu", "медеуский": "Medeu",
    "nauryzbay": "Nauryzbay", "nauryzbayskiy": "Nauryzbay", "наурызбайский": "Nauryzbay",
    "turksib": "Turksib", "turksibskiy": "Turksib", "турксибский": "Turksib",
    "zhetysu": "Zhetysu", "zhetysuskiy": "Zhetysu", "жетысуский": "Zhetysu",
}

df["district"] = df["district"].map(mapping)

print("nunique:", df["district"].nunique())
print("NaN:", df["district"].isna().sum())
print(df["district"].value_counts())

nunique: 8
NaN: 0
district
Medeu        153
Bostandyk    147
Auezov       144
Turksib      141
Zhetysu      129
Almaly       124
Alatau       121
Nauryzbay    121
Name: count, dtype: int64


тут привели все варианты написания районов к единому виду. Убрали пробелы, а mapping заменил 24 разных варианта на 8 канонических названий (Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu). 

In [13]:
for col in ["rooms", "area_m2", "price_kzt"]:
    print("===", col, "===")
    print(df[col].dropna().sample(10, random_state=YOUR_ID).tolist())

=== rooms ===
['3', '2', '5', 'studio', '3', '2', '3', '2', '2', '3']
=== area_m2 ===
['73.1', '61.1 m²', '156.6', '32.4', '78.1', '55.5', '84.4 m²', '52.7', '53.7', '75.6']
=== price_kzt ===
['36070000', '62 020 000 ₸', '162880000', '23030000', '84310000', '62610000', '82 080 000 ₸', '56000000', '38290000', '49100000']


тут видим что rooms сщдержит текст вместе с числами, а в area_m2 нескольо значений с суфиксом м^2 в price_kzt есть числа с пробелами

In [14]:
df["rooms"] = df["rooms"].replace("studio", 1)
df["rooms"] = pd.to_numeric(df["rooms"], errors="coerce")
print("rooms NaN:", df["rooms"].isna().sum())


rooms NaN: 0


заменили студио на число, а потом через pd.to_numeric превратили всю колонку в числа. Результат NaN: 0 означает, что все значения стали числовыми — задача по rooms выполнена.

In [15]:
df["area_m2"] = (
    df["area_m2"].astype(str)
      .str.replace("m²", "", regex=False)
      .str.strip()
      .str.replace(",", ".", regex=False)
)
df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
print("area_m2 NaN:", df["area_m2"].isna().sum())

area_m2 NaN: 0


In [16]:
df["price_kzt"] = (
    df["price_kzt"].astype(str)
      .str.replace(r"\D", "", regex=True)
)
df["price_kzt"] = pd.to_numeric(df["price_kzt"], errors="coerce")
print("price_kzt NaN:", df["price_kzt"].isna().sum())

price_kzt NaN: 0


In [17]:
print(df[["rooms", "area_m2", "price_kzt"]].dtypes)
print(df["district"].nunique())
print("shape:", df.shape)

rooms          int64
area_m2      float64
price_kzt      int64
dtype: object
8
shape: (1080, 11)


Все три колонки приведены к числам. В area_m2 убрали суффикс, пробелы и заменили запятую на точку, потом через to_numeric превратили в float. В price_kzt удалили всё, кроме цифр (запятые, пробелы, символ тенге), и тоже превратили в число. 

Итог : 8 районов, все числа распарсены, shape = 1080 × 11.